In [1]:
%pip install lightgbm xgboost catboost


  Using cached lightgbm-4.6.0-py3-none-manylinux_2_28_x86_64.whl (3.6 MB)
  Using cached xgboost-3.0.2-py3-none-manylinux_2_28_x86_64.whl (253.9 MB)
  Using cached catboost-1.2.8-cp310-cp310-manylinux2014_x86_64.whl (99.2 MB)
  Using cached nvidia_nccl_cu12-2.26.5-py3-none-manylinux2014_x86_64.manylinux_2_17_x86_64.whl (318.1 MB)
  Using cached plotly-6.1.1-py3-none-any.whl (16.1 MB)
  Using cached graphviz-0.20.3-py3-none-any.whl (47 kB)
  Using cached narwhals-1.41.0-py3-none-any.whl (357 kB)
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import pandas as pd
from src.config import *
from src.utils import get_latest_file
from predictions import *

from sklearn.ensemble import RandomForestClassifier, StackingClassifier
from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
pd.set_option("display.max_columns", None)



In [6]:

# Charger le dernier fichier d'équipe
teams_file = get_latest_file(DATA_TEAMS_DIR)
teams_df = pd.read_csv(teams_file)

# Afficher les noms disponibles pour aide à la recherche
print("Liste des équipes disponibles:")
print(teams_df['full_name'].unique())

# Rechercher les IDs des équipes désirées
# target_teams = ['Indiana Pacers', 'New York Knicks']

# team_home_name = 'Minnesota Timberwolves'
# team_away_name = 'Oklahoma City Thunder'

team_home_name = 'Indiana Pacers'
team_away_name = 'New York Knicks'

team_home_id = teams_df[teams_df['full_name'] == team_home_name]['id'].values[0]
team_away_id = teams_df[teams_df['full_name'] == team_away_name]['id'].values[0]
print(f"ID de l'équipe {team_home_name}: {team_home_id}")
print(f"ID de l'équipe {team_away_name}: {team_away_id}")

# filtered = teams_df[teams_df['full_name'].isin(target_teams)]
# print("\nIDs des équipes sélectionnées:")
# print(filtered[['full_name', 'id']])


Liste des équipes disponibles:
['Atlanta Hawks' 'Boston Celtics' 'Cleveland Cavaliers'
 'New Orleans Pelicans' 'Chicago Bulls' 'Dallas Mavericks'
 'Denver Nuggets' 'Golden State Warriors' 'Houston Rockets'
 'Los Angeles Clippers' 'Los Angeles Lakers' 'Miami Heat'
 'Milwaukee Bucks' 'Minnesota Timberwolves' 'Brooklyn Nets'
 'New York Knicks' 'Orlando Magic' 'Indiana Pacers' 'Philadelphia 76ers'
 'Phoenix Suns' 'Portland Trail Blazers' 'Sacramento Kings'
 'San Antonio Spurs' 'Oklahoma City Thunder' 'Toronto Raptors' 'Utah Jazz'
 'Memphis Grizzlies' 'Washington Wizards' 'Detroit Pistons'
 'Charlotte Hornets']
ID de l'équipe Indiana Pacers: 1610612754
ID de l'équipe New York Knicks: 1610612752


In [7]:
# 🔁 Charger le dataset complet
dataset_path = get_latest_file(DATA_FINAL_CLEANED_DATASET_DIR)
full_df = pd.read_csv(dataset_path)
print("Loaded dataset:", dataset_path)

predicted_rows = build_prediction_rows(home_team_id=team_home_id, away_team_id=team_away_id, dataset=full_df)

predicted_rows


Loaded dataset: data/final_cleaned_dataset/nba_features_cleaned_final_2025-05-26_17-54-31.csv


,GAME_ID,TEAM_ID,GAME_DATE,OPP_TEAM_ID,SEASON,IS_HOME,IS_WIN,ROLL_HOME_WINRATE_3,ROLL_AWAY_WINRATE_3,ROLL_HOME_WINRATE_5,ROLL_AWAY_WINRATE_5,ROLL_HOME_WINRATE_10,ROLL_AWAY_WINRATE_10,ROLL_HOME_WINRATE_25,ROLL_AWAY_WINRATE_25,ROLL_HOME_WINRATE_50,ROLL_AWAY_WINRATE_50,ROLL_HOME_WINRATE_100,ROLL_AWAY_WINRATE_100,ROLL_HOME_WINRATE_200,ROLL_AWAY_WINRATE_200,ROLL_WIN_RATIO_3,ROLL_WIN_RATIO_5,ROLL_WIN_RATIO_10,ROLL_WIN_RATIO_25,ROLL_WIN_RATIO_50,ROLL_WIN_RATIO_100,ROLL_WIN_RATIO_200,WIN_STREAK,HOME_WIN_STREAK,AWAY_WIN_STREAK,DAYS_SINCE_LAST_GAME,OPP_DAYS_SINCE_LAST_GAME,REST_ADVANTAGE,ROLL_HOME_REST_ADV_3,ROLL_AWAY_REST_ADV_3,ROLL_HOME_REST_ADV_5,ROLL_AWAY_REST_ADV_5,ROLL_HOME_REST_ADV_10,ROLL_AWAY_REST_ADV_10,ROLL_HOME_REST_ADV_25,ROLL_AWAY_REST_ADV_25,ROLL_HOME_REST_ADV_50,ROLL_AWAY_REST_ADV_50,ROLL_HOME_REST_ADV_100,ROLL_AWAY_REST_ADV_100,ROLL_HOME_REST_ADV_200,ROLL_AWAY_REST_ADV_200,H2H_LAST_3_DIFF,H2H_LAST_3_WINRATE,H2H_LAST_3_COUNT,H2H_LAST_5_DIFF,H2H_LAST_5_WINRATE,H2H_LAST_5_COUNT,H2H_LAST_10_DIFF,H2H_LAST_10_WINRATE,H2H_LAST_10_COUNT,H2H_LAST_25_DIFF,H2H_LAST_25_WINRATE,H2H_LAST_25_COUNT,H2H_LAST_50_DIFF,H2H_LAST_50_WINRATE,H2H_LAST_50_COUNT,H2H_LAST_100_DIFF,H2H_LAST_100_WINRATE,H2H_LAST_100_COUNT,H2H_LAST_200_DIFF,H2H_LAST_200_WINRATE,H2H_LAST_200_COUNT,H2H_SEASON_WINS,H2H_SEASON_MATCHES,H2H_SEASON_WINRATE,H2H_WIN_STREAK,ELO_PRE,OPP_ELO_PRE,ELO_PRE_SEASON,OPP_ELO_PRE_SEASON
64148,NaN,1610612752,2025-05-26,1610612754,2024-25,0,0,0.0,1.0,0.333333,0.5,0.4,0.8,0.533333,0.8,0.615385,0.625000,0.627451,0.591837,0.653465,0.565657,0.333333,0.4,0.6,0.64,0.62,0.61,0.610,1,0,1,1.0,1.0,0.0,-47.0,0.0,-31.333333,0.000000,-18.8,-4.600000,-41.200000,-18.4,-90.153846,-71.500,-94.176471,-110.979592,-92.722772,-117.242424,-1,0.333333,3,-1,0.4,5,-2,0.4,10,1,0.52,25,-10,0.4,50,-20,0.4,100,-21,0.4,105,3,6,0.5,1,1630.661757,1672.724168,1623.421258,1668.412456
64149,NaN,1610612754,2025-05-26,1610612752,2024-25,1,0,0.0,1.0,0.500000,1.0,0.5,1.0,0.733333,0.8,0.769231,0.583333,0.687500,0.519231,0.656566,0.495050,0.666667,0.8,0.8,0.76,0.68,0.60,0.575,0,0,5,1.0,1.0,0.0,0.0,-45.5,0.000000,-30.333333,0.0,-18.333333,-33.933333,-28.2,-70.000000,-95.125,-75.312500,-125.826923,-94.858586,-116.366337,1,0.666667,3,1,0.6,5,2,0.6,10,-1,0.48,25,10,0.6,50,20,0.6,100,21,0.6,105,3,6,0.5,0,1672.724168,1630.661757,1668.412456,1623.421258


In [4]:
# 🔁 Charger le dataset complet
dataset_path = get_latest_file(DATA_FINAL_CLEANED_DATASET_DIR)
full_df = pd.read_csv(dataset_path)
print("Loaded dataset:", dataset_path)

predicted_rows = build_prediction_rows(home_team_id=team_home_id, away_team_id=team_away_id, dataset=full_df)

predicted_rows


Loaded dataset: data/final_cleaned_dataset/nba_features_cleaned_final_2025-05-26_17-54-31.csv


,GAME_ID,TEAM_ID,GAME_DATE,OPP_TEAM_ID,SEASON,IS_HOME,IS_WIN,ROLL_HOME_WINRATE_3,ROLL_AWAY_WINRATE_3,ROLL_HOME_WINRATE_5,ROLL_AWAY_WINRATE_5,ROLL_HOME_WINRATE_10,ROLL_AWAY_WINRATE_10,ROLL_HOME_WINRATE_25,ROLL_AWAY_WINRATE_25,ROLL_HOME_WINRATE_50,ROLL_AWAY_WINRATE_50,ROLL_HOME_WINRATE_100,ROLL_AWAY_WINRATE_100,ROLL_HOME_WINRATE_200,ROLL_AWAY_WINRATE_200,ROLL_WIN_RATIO_3,ROLL_WIN_RATIO_5,ROLL_WIN_RATIO_10,ROLL_WIN_RATIO_25,ROLL_WIN_RATIO_50,ROLL_WIN_RATIO_100,ROLL_WIN_RATIO_200,WIN_STREAK,HOME_WIN_STREAK,AWAY_WIN_STREAK,DAYS_SINCE_LAST_GAME,OPP_DAYS_SINCE_LAST_GAME,REST_ADVANTAGE,ROLL_HOME_REST_ADV_3,ROLL_AWAY_REST_ADV_3,ROLL_HOME_REST_ADV_5,ROLL_AWAY_REST_ADV_5,ROLL_HOME_REST_ADV_10,ROLL_AWAY_REST_ADV_10,ROLL_HOME_REST_ADV_25,ROLL_AWAY_REST_ADV_25,ROLL_HOME_REST_ADV_50,ROLL_AWAY_REST_ADV_50,ROLL_HOME_REST_ADV_100,ROLL_AWAY_REST_ADV_100,ROLL_HOME_REST_ADV_200,ROLL_AWAY_REST_ADV_200,H2H_LAST_3_DIFF,H2H_LAST_3_WINRATE,H2H_LAST_3_COUNT,H2H_LAST_5_DIFF,H2H_LAST_5_WINRATE,H2H_LAST_5_COUNT,H2H_LAST_10_DIFF,H2H_LAST_10_WINRATE,H2H_LAST_10_COUNT,H2H_LAST_25_DIFF,H2H_LAST_25_WINRATE,H2H_LAST_25_COUNT,H2H_LAST_50_DIFF,H2H_LAST_50_WINRATE,H2H_LAST_50_COUNT,H2H_LAST_100_DIFF,H2H_LAST_100_WINRATE,H2H_LAST_100_COUNT,H2H_LAST_200_DIFF,H2H_LAST_200_WINRATE,H2H_LAST_200_COUNT,H2H_SEASON_WINS,H2H_SEASON_MATCHES,H2H_SEASON_WINRATE,H2H_WIN_STREAK,ELO_PRE,OPP_ELO_PRE,ELO_PRE_SEASON,OPP_ELO_PRE_SEASON
64148,NaN,1610612750,2025-05-26,1610612760,2024-25,1,0,1.0,0.0,1.0,0.333333,0.800000,0.60,0.769231,0.583333,0.720000,0.60000,0.591837,0.568627,0.653061,0.598039,0.333333,0.6,0.7,0.68,0.66,0.58,0.625,1,1,0,2.0,2.0,0.0,0.0,-39.5,0.000000,-26.333333,-21.000000,-15.8,-56.076923,-52.083333,-82.000000,-59.320000,-109.510204,-95.333333,-104.928571,-103.872549,-1,0.333333,3,-1,0.4,5,-2,0.4,10,3,0.56,25,-4,0.46,50,-18,0.41,100,-17,0.415842,101,3,7,0.428571,1,1663.574626,1767.766450,1648.465100,1747.430693
64149,NaN,1610612760,2025-05-26,1610612750,2024-25,0,0,1.0,0.0,1.0,0.000000,0.833333,0.25,0.866667,0.600000,0.851852,0.73913,0.826923,0.729167,0.810000,0.650000,0.666667,0.6,0.6,0.76,0.80,0.78,0.730,0,2,0,2.0,2.0,0.0,-41.5,0.0,-27.666667,0.000000,-21.666667,0.0,-42.066667,-22.000000,-57.259259,-54.434783,-110.576923,-96.937500,-120.200000,-97.050000,1,0.666667,3,1,0.6,5,2,0.6,10,-3,0.44,25,4,0.54,50,18,0.59,100,17,0.584158,101,4,7,0.571429,0,1767.766450,1663.574626,1747.430693,1648.465100


In [8]:
import joblib

#open last stacking model from DATA_MODELS_DIR 
stacking_model_path = get_latest_file(DATA_MODELS_DIR)
print("Loaded stacking model:", stacking_model_path)

# Charger le modèle de stacking
pipeline = joblib.load(stacking_model_path)


# Colonnes à ignorer
drop_cols = ['TEAM_ID','OPP_TEAM_ID','SEASON','GAME_DATE','IS_WIN']
X_pred = predicted_rows.drop(columns=drop_cols, errors='ignore')

# ⚠️ Réordonner les colonnes si besoin
X_pred = X_pred[[col for col in pipeline.named_steps['scaler'].get_feature_names_out() if col in X_pred.columns]]

# Prédictions
pred_classes = pipeline.predict(X_pred)
pred_probas = pipeline.predict_proba(X_pred)[:, 1]

# Résultats
results_df = predicted_rows[['TEAM_ID', 'IS_HOME']].copy()
results_df['PREDICTED_WIN'] = pred_classes
results_df['WIN_PROBA'] = pred_probas

# Affichage lisible avec nom d’équipe
results_df = results_df.merge(teams_df[['id', 'full_name']], left_on='TEAM_ID', right_on='id', how='left')
results_df = results_df[['full_name', 'IS_HOME', 'PREDICTED_WIN', 'WIN_PROBA']].rename(columns={'full_name': 'TEAM'})

display(results_df)

Loaded stacking model: data/models/stacking_model_2025-05-26.joblib


,TEAM,IS_HOME,PREDICTED_WIN,WIN_PROBA
0,New York Knicks,0,0,0.349225
1,Indiana Pacers,1,1,0.636354
